In [158]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

import plotly.express as px
import plotly.io as pio 

from dsc80_utils import * 

## Introduction

This project uses two Spotify related datasets: `music_tracks.csv` and `artists.csv`. 

In [159]:
artists = pd.read_csv("data/artists.csv")
music_tracks = pd.read_csv("data/music_tracks.csv")

The `music_tracks.csv` dataset contains information for 114,000 tracks across 114 genres. Each row represents a single track, and includes both metadata and Spotify-generated audio features. These features include variables such as danceability, energy, acousticness, valence, tempo, explicit content, and popularity score. The `artists.csv` dataset contains artist-level information, including artist popularity, follower counts, and genre tags.

In this project, the primary focus is on the `music_tracks.csv` dataset because it contains track-level audio characteristics that may help explain why some songs become more popular than others.

The main research question I want to explore is: <br><br>
**Can we predict whether a Spotify song becomes popular based on its audio features and genre?** 

This question helps us better understand how audio characteristics and certain genres are associated with successful songs, providing insights into modern music trends and listener preferences. 

In [160]:
music_tracks.head()

,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,release_date,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,1974,False,0.68,0.46,1,-6.75,0,0.14,0.03,1.01e-06,0.36,0.71,87.92,4,acoustic
1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,1995-04,False,0.42,0.17,1,-17.23,1,0.08,0.92,5.56e-06,0.10,0.27,77.49,4,acoustic
2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,1973,False,0.44,0.36,0,-9.73,1,0.06,0.21,0.00e+00,0.12,0.12,76.33,4,acoustic
3,3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,2018-08-10,False,0.27,0.06,0,-18.52,1,0.04,0.91,7.07e-05,0.13,0.14,181.74,3,acoustic
4,4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,2017-02-03,False,0.62,0.44,2,-9.68,1,0.05,0.47,0.00e+00,0.08,0.17,NaN,4,acoustic


In [161]:
music_tracks.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 114000 entries, 0 to 113999
Data columns (total 22 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   Unnamed: 0        114000 non-null  int64  
 1   track_id          114000 non-null  object 
 2   artists           113999 non-null  object 
 3   album_name        113999 non-null  object 
 4   track_name        113999 non-null  object 
 5   popularity        114000 non-null  int64  
 6   duration_ms       114000 non-null  int64  
 7   release_date      114000 non-null  object 
 8   explicit          114000 non-null  bool   
 9   danceability      114000 non-null  float64
 10  energy            114000 non-null  float64
 11  key               114000 non-null  int64  
 12  loudness          114000 non-null  float64
 13  mode              114000 non-null  int64  
 14  speechiness       114000 non-null  float64
 15  acousticness      114000 non-null  float64
 16  instrumentalness  11

In [162]:
artists.head()

,id,followers,genres,name,popularity
0,0DheY5irMjBUeLybbCUEZ2,0.0,[],Armid & Amir Zare Pashai feat. Sara Rouzbehani,0
1,0DlhY15l3wsrnlfGio2bjU,5.0,[],ปูนา ภาวิณี,0
2,0DmRESX2JknGPQyO15yxg7,0.0,[],Sadaa,0
3,0DmhnbHjm1qw6NCYPeZNgJ,0.0,[],Tra'gruda,0
4,0Dn11fWM7vHQ3rinvWEl4E,2.0,[],Ioannis Panoutsopoulos,0


In [163]:
artists.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1162095 entries, 0 to 1162094
Data columns (total 5 columns):
 #   Column      Non-Null Count    Dtype  
---  ------      --------------    -----  
 0   id          1162095 non-null  object 
 1   followers   1162084 non-null  float64
 2   genres      1162095 non-null  object 
 3   name        1162092 non-null  object 
 4   popularity  1162095 non-null  int64  
dtypes: float64(1), int64(1), object(3)
memory usage: 44.3+ MB


In [164]:
print(music_tracks.shape)
print(artists.shape)

(114000, 22)
(1162095, 5)


In [165]:
music_tracks.describe()

,Unnamed: 0,popularity,duration_ms,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature
count,114000.00,114000.00,1.14e+05,114000.00,114000.00,114000.00,114000.00,114000.00,114000.00,114000.00,1.14e+05,114000.00,114000.00,91886.00,114000.00
mean,56999.50,33.24,2.28e+05,0.57,0.64,5.31,-8.26,0.64,0.08,0.31,1.56e-01,0.21,0.47,123.12,3.90
std,32909.11,22.31,1.07e+05,0.17,0.25,3.56,5.03,0.48,0.11,0.33,3.10e-01,0.19,0.26,29.78,0.43
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
50%,56999.50,35.00,2.13e+05,0.58,0.69,5.00,-7.00,1.00,0.05,0.17,4.16e-05,0.13,0.46,123.00,4.00
75%,85499.25,50.00,2.62e+05,0.69,0.85,8.00,-5.00,1.00,0.08,0.60,4.90e-02,0.27,0.68,141.65,4.00
max,113999.00,100.00,5.24e+06,0.98,1.00,11.00,4.53,1.00,0.96,1.00,1.00e+00,1.00,0.99,222.60,5.00


In [166]:
artists.describe()

,followers,popularity
count,1.16e+06,1.16e+06
mean,1.02e+04,8.80e+00
std,2.54e+05,1.36e+01
...,...,...
50%,5.70e+01,2.00e+00
75%,4.17e+02,1.30e+01
max,7.89e+07,1.00e+02


## Data Cleaning and EDA

In [167]:
# Check missing values
music_tracks.isnull().sum().sort_values(ascending=False)

tempo          22114
artists            1
album_name         1
               ...  
duration_ms        0
popularity         0
track_genre        0
Length: 22, dtype: int64

The `tempo` column contains over 22,000 missing values, which may affect the reliability of analyses involving tempo. Since tempo is an important audio feature, I preserve these missing values for later missingness analysis rather then removing the rows immediately. 

In [168]:
cleaned_tracks = music_tracks.drop(columns=["Unnamed: 0"])

The `Unnamed: 0` column was removed because it was duplicated with the index column and did not contain any meaningful information for analysis. 

In [169]:
cleaned_tracks.head()

,track_id,artists,album_name,track_name,popularity,duration_ms,release_date,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,1974,False,0.68,0.46,1,-6.75,0,0.14,0.03,1.01e-06,0.36,0.71,87.92,4,acoustic
1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,1995-04,False,0.42,0.17,1,-17.23,1,0.08,0.92,5.56e-06,0.10,0.27,77.49,4,acoustic
2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,1973,False,0.44,0.36,0,-9.73,1,0.06,0.21,0.00e+00,0.12,0.12,76.33,4,acoustic
3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,2018-08-10,False,0.27,0.06,0,-18.52,1,0.04,0.91,7.07e-05,0.13,0.14,181.74,3,acoustic
4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,2017-02-03,False,0.62,0.44,2,-9.68,1,0.05,0.47,0.00e+00,0.08,0.17,NaN,4,acoustic


In [170]:
# Remove duplicates
print('Number of duplicate rows:', cleaned_tracks.duplicated().sum())
cleaned_tracks = cleaned_tracks.drop_duplicates()


Number of duplicate rows: 54


I identified and dropped the 54 duplicate rows to avoid repeated tracks affecting the analysis. 

In [171]:
cleaned_tracks['release_date']

0               1974
1            1995-04
2               1973
             ...    
113997    1992-10-21
113998          1992
113999          1981
Name: release_date, Length: 113946, dtype: object

In [172]:
# Convert release date
cleaned_tracks['release_date'] = pd.to_datetime(
    cleaned_tracks['release_date'].astype(str),
    format='mixed',
    errors = 'coerce'
)

# Extract year
cleaned_tracks['release_year'] = cleaned_tracks['release_date'].dt.year

The `release_date` column originally contained inconsistent date formats. To keep the data consistent, I converted the column to datetime format and extracted a new `release_year` feature to support more efficient analysis. 

In [173]:
# Convert duration into minutes 
cleaned_tracks['duration_min'] = (
    cleaned_tracks['duration_ms'] / 60000
).round(2)

I created a `duration_min` feature by converting milliseconds into minutes, since minutes are more intuitive for understanding song length patterns and comparing tracks. 

In [174]:
# Create decade feature 
cleaned_tracks['decade'] = (
    cleaned_tracks['release_year'] // 10
) * 10

I also grouped songs by decade using the `release_year` column to better examine long-term trends in music popularity over time.

In [175]:
cleaned_tracks['num_artists'] = (
    cleaned_tracks['artists']
    .str.split(';')
    .str.len()
)

A new feature `num_artists` was also created to count the number of artists credited on each track by splitting the `artists` column using semicolumns. This may help examine whether collaborations between multiple artists are associated with higher song popularity.

In [176]:
# Inspect popularity distribution 
cleaned_tracks['popularity'].describe()

count    113946.00
mean         33.24
std          22.30
           ...    
50%          35.00
75%          50.00
max         100.00
Name: popularity, Length: 8, dtype: float64

In [177]:
# Finding threshold for popularity score

print((cleaned_tracks["popularity"] >= 50).value_counts(normalize=True))

print((cleaned_tracks["popularity"] >= 60).value_counts(normalize=True))

print((cleaned_tracks["popularity"] >= 70).value_counts(normalize=True))

print((cleaned_tracks["popularity"] >= 55).value_counts(normalize=True))

popularity
False    0.74
True     0.26
Name: proportion, dtype: float64
popularity
False    0.87
True     0.13
Name: proportion, dtype: float64
popularity
False    0.95
True     0.05
Name: proportion, dtype: float64
popularity
False    0.81
True     0.19
Name: proportion, dtype: float64


In [178]:
cleaned_tracks['is_popular'] = (
    cleaned_tracks['popularity'] >= 55
)

Because this project involves classification, I tested several popularity thresholds and compared their class distributions. I selected 55 as the popularity threshold because it produced the most balanced class distribution out of all the tested thresholds while still representing moderately popluar songs. 

### Univariate Analysis

In [179]:
# Popularity distribution
fig = px.histogram(
    cleaned_tracks,
    x='popularity',
    nbins=20,
    title='Popularity Distribution'
)

fig.show()

The popularity distribution is right-skewed, which suggests that most songs are relativly less popular and that highly popular songs are less common in the dataset. 

In [180]:
# Danceability Distribution
fig = px.histogram(
    cleaned_tracks,
    x='danceability',
    nbins=20,
    title='Danceability Distribution'
)

fig.show()

From this danceability distribution, I observed that most songs have moderate-to-high danceability scores, indicating that Spotify tracks in this dataset tend to have rhythmically engaging music. 

In [181]:
# Top 10 most common genres
top_genres = (
    cleaned_tracks['track_genre']
    .value_counts()
    .head(10)
)

px.bar(
    top_genres,
    title='Top 10 Most Common Genres'
)

This barchart shows the top 10 most common genres in the dataset. The frequences of these genres appear to be very similar, with each genre containing close to 1000 tracks. This suggests that the dataset is relatively balanced across genres rather than being dominated by a few. Having a balanced distribution is crucial for conducting fair analysis and predictive modeling since it reduces the risk of bias toward a specific genre. 

In [182]:
# Explicit vs Non Explicit counts
px.histogram(
    cleaned_tracks,
    x='explicit',
    title='Explicit vs Non-Explicit Tracks'
)

This chart reveals a significant difference between explicit and non-explicit tracks in the dataset. It appears that Non-explicit songs make up the vast majority of tracks, with over 100,000 entries, while explicit songs account for only a much smaller portion of the dataset. This suggests that mainstream music in the dataset is still largely dominated by non-explicit content. This imbalance may also affect later analysis or predictive modeling, where patterns associated with non-explicit tracks could have a stronger influence on the model due to their much larger representation in the dataset.

### Bivariate Analysis

In [183]:
# Danceability vs Popularity 
fig = px.scatter(
    cleaned_tracks,
    x='danceability',
    y='popularity',
    title='Danceability vs Popularity',
    opacity=0.2
)

fig.show()

The scatterplot suggests a weak positive relationship between popularity and danceability. Songs with higher danceability appear to be more popular, however the relationship is not strongly linear. 

In [184]:
# Energy vs Popluarity
fig = px.scatter(
    cleaned_tracks,
    x='energy',
    y='popularity',
    title='Energy vs Popularity',
    opacity=0.2
)

fig.show()

The scatterplot does not show a strong linear relationship between energy and popularity, suggesting that energy alone may not strongly predict a song's popularity. However, moderately energetic songs appear more common among higher popularity scores. 

In [185]:
# Danceability by Popularity class
fig = px.box(
    cleaned_tracks,
    x='is_popular',
    y='danceability',
    color='is_popular',
    title='Danceability by Popularity Class'
)

fig.show()

From this boxplot, there seems to be not much difference in danceability between popular and non-popular songs. Both groups have a similar median values and distributions, suggesting that danceability alone may not be a strong predictor of popularity. However, popular songs appear to have higher median danceability score overall. 

In [186]:
# Explicit vs Popularity
px.box(
    cleaned_tracks,
    x='explicit',
    y='popularity',
    title='Popularity by Explicit Content'
)

Interestingly, although non-explicit tracks make up the majority of the dataset, explicit tracks appear to have a slightly higher median popularity. This may suggest that explicit content is relatively common among more popular or commercially successful tracks. However, both groups still show a wide spread in popularity, indicating that explicitness alone is not enough to determine whether a song becomes popular.

In [187]:
# Release Year vs Popularity
px.scatter(
    cleaned_tracks.sample(5000),
    x='release_year',
    y='popularity',
    opacity=0.3,
    title='Release Year vs Popularity'
)

While there is no strong correlation between release year and popularity, more recent songs appear to have slightly higher popularity scores on average compared to older tracks. In addition, many songs across all years have popularity values near 0, suggesting that a large portion of tracks in the dataset are relatively unpopular.

In [188]:
# Valence vs Popularity
px.scatter(
    cleaned_tracks.sample(5000), 
    x='valence',
    y='popularity',
    opacity=0.3,
    title='Valence vs Popularity'
)

Overall, there is no strong correlation between valence and popularity, which suggests that songs with both low and high valence levels can achieve a wide range of popularity scores. However, moderately popular songs seem to appear slightly more frequently across mid-range valence values. The plot also shows many songs with popularity scores near 0 across all valence levels, suggesting that a song’s emotional tone alone is not a strong predictor of popularity.

In [189]:
# Correlation Heatmap
audio_cols = [
    'popularity',
    'danceability',
    'energy',
    'speechiness',
    'acousticness',
    'instrumentalness',
    'liveness',
    'valence',
    'tempo'
]

corr = cleaned_tracks[audio_cols].corr()

fig = px.imshow(
    corr, 
    text_auto=True,
    color_continuous_scale='blues',
    title='Correlation Matrix of Audio Features'
)

fig.show()

The correlation matrix shows the relationships between popularity and several audio features. Overall, popularity has relatively weak correlations with most features, suggesting that no single audio characteristic strongly determines whether a song becomes popular.

### Interesting Aggregates

In [190]:
# To see which genres tend to have the highest average popularity
genre_popularity = (
    cleaned_tracks.groupby('track_genre')['popularity']
    .mean()
    .sort_values(ascending=False)
)
genre_popularity

track_genre
pop-film    59.28
k-pop       56.90
chill       53.65
            ...  
latin        8.31
romance      3.25
iranian      2.21
Name: popularity, Length: 114, dtype: float64

In [191]:
fig = px.bar(
    genre_popularity.head(10),
    title='Top Genres by Average Popularity'
)

fig.show()

Certain genres such as pop-film and k-pop have noticeably higher average popularity scores compared to other genres. This suggests that genere plays an important role in predicting song popularity. 

In [192]:
# How do audio features differ between popular and non-popular songs

audio_feature_means = (
    cleaned_tracks.groupby('is_popular')[
        [
            'danceability',
            'energy',
            'loudness',
            'speechiness',
            'acousticness',
            'instrumentalness',
            'liveness',
            'valence',
            'tempo'
        ]
    ]
    .mean()
)

pd.set_option('display.max_columns', None)
audio_feature_means

,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo
is_popular,,,,,,,,,
False,0.56,0.64,-8.36,0.09,0.32,0.17,0.22,0.48,123.43
True,0.59,0.64,-7.82,0.08,0.29,0.10,0.18,0.47,121.79


The table compares the average audio features between popular and non-popular songs. Popular songs tend to have slightly higher average danceability scores and higher loudness values, suggesting that more rhythmically engaging and louder tracks may perform better on Spotify. Popular songs also show lower average acousticness and instrumentalness, indicating that mainstream songs in the dataset are generally less acoustic and more vocal-focused.

However, many of the differences between the two groups remain relatively small, especially for features such as energy, speechiness, and valence. This suggests that no single audio feature alone strongly determines popularity, and that song popularity is likely influenced by a combination of multiple audio characteristics and genre.

## Assessment of Missingness

In [193]:
cleaned_tracks.isna().sum().sort_values(ascending=False)

tempo          22111
album_name         1
track_name         1
               ...  
duration_ms        0
popularity         0
is_popular         0
Length: 26, dtype: int64

In [194]:
cleaned_tracks['tempo_missing'] = cleaned_tracks['tempo'].isna()

I selected `tempo` as the column to analyze because it has substantial number of missing values compared to other columns. Then, I created a boolean indicator column, `tempo_missing`, to represent whether each track has a missing tempo value.

I would first pick `track_genre` to check if it is likely related to tempo missingess, since different genres have different musical structures, which may make tempo detection harder. 

In [195]:
(
    cleaned_tracks
    .groupby('track_genre')['tempo_missing']
    .mean()
    .sort_values(ascending=False)
    .head(15)
)

track_genre
romance     0.37
ambient     0.37
new-age     0.35
            ... 
chill       0.30
acoustic    0.30
tango       0.29
Name: tempo_missing, Length: 15, dtype: float64

To explore whether tempo missingness may depend on genre, I calculated the proportion of missing tempo values within each genre. Genres such as romance, ambient and new-age have noticeably higher missingness rates, suggesting that genre may be related to whether a tempo value is missing.

### Permutation Test 1: check if tempo missingness depends on track genre

Null Hypothesis: The missingness of `tempo` does not depend on `track_genre`. 
<br>
Alternative Hypothesis: The missingness of `tempo` depends on `track_genre`. 

Test Statistic: Total Variation Distance (TVD) 

In [196]:
def tvd(data, group_col, cat_col):

    props = (
        data
        .pivot_table(
            index=cat_col,
            columns=group_col,
            aggfunc='size',
            fill_value=0
        )
    )

    props = props / props.sum()

    return 0.5 * np.abs(
        props[True] - props[False]
    ).sum()


# Compute Observed TVD
observed_tvd = tvd(
    cleaned_tracks,
    'tempo_missing',
    'track_genre'
)

observed_tvd

np.float64(0.17900235126090122)

In [197]:
# Permutation Test
tvds = []

for _ in range(1000):

    shuffled = cleaned_tracks.copy()

    shuffled['tempo_missing'] = np.random.permutation(
        shuffled['tempo_missing']
    )

    tvds.append(
        tvd(
            shuffled,
            'tempo_missing',
            'track_genre'
        )
    )

tvds = np.array(tvds)

# Compute P-value
p_value = np.mean(tvds >= observed_tvd)

p_value

np.float64(0.0)

In [198]:
# Visualization
fig = px.histogram(
    x=tvds,
    nbins=30,
    title=f'Permutation Distribution (p = {p_value})'
)

fig.add_vline(
    x=observed_tvd,
    line_color='red',
    line_width=3,
    annotation_text='Observed TVD'
)

fig.show()

Since the observed TVD lies far outside the permuatation and the p-value is approximately 0, therefore I reject the null hypothesis and conclude that the missingness of `tempo` depends on `track_genre`.

In [199]:
cleaned_tracks.columns

Index(['track_id', 'artists', 'album_name', 'track_name', 'popularity',
       'duration_ms', 'release_date', 'explicit', 'danceability', 'energy',
       'key', 'loudness', 'mode', 'speechiness', 'acousticness',
       'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature',
       'track_genre', 'release_year', 'duration_min', 'decade', 'num_artists',
       'is_popular', 'tempo_missing'],
      dtype='object')

### Permutation Test 2: check if tempo missingness depends on release year

Null Hypothesis: The missingness of `tempo` does not depend on `release_year`. 
<br>
Alternative Hypothesis: The missingness of `tempo` depends on `release_year`. 

Test Statistic: Absolute Difference in Means

In [200]:
# Observed Statistic
observed_diff = abs(
    cleaned_tracks
    .groupby('tempo_missing')['release_year']
    .mean()
    .diff()
    .iloc[-1]
)

observed_diff

np.float64(1.2095377619245937)

In [201]:
# Permutation Test
diffs = []

for _ in range(1000):

    shuffled = cleaned_tracks.copy()

    shuffled['tempo_missing'] = np.random.permutation(
        shuffled['tempo_missing']
    )

    diff = abs(
        shuffled
        .groupby('tempo_missing')['release_year']
        .mean()
        .diff()
        .iloc[-1]
    )

    diffs.append(diff)

diffs = np.array(diffs)

# Compute P-value
p_value = np.mean(
    diffs >= observed_diff
)

p_value

np.float64(0.0)

In [202]:
# Visualization
fig = px.histogram(
    x=np.append(diffs, observed_diff),
    nbins=30,
    title=f'Permutation Distribution (p = {p_value})'
)

fig.add_vline(
    x=observed_diff,
    line_color='red',
    line_width=3,
    annotation_text='Observed Difference'
)

fig.show()

In [203]:
cols = [
    'mode', 'time_signature', 'danceability',
    'energy', 'liveness', 'valence',
    'speechiness', 'instrumentalness',
    'duration_ms'
]

for col in cols:
    
    observed_diff = abs(
        cleaned_tracks
        .groupby('tempo_missing')[col]
        .mean()
        .diff()
        .iloc[-1]
    )
    
    print(col, observed_diff)

mode 0.02401522412728463
time_signature 0.05175574732097088
danceability 0.012553871890651824
energy 0.15912073741872768
liveness 0.02400669603268807
valence 0.039671320041122005
speechiness 0.010692377314873519
instrumentalness 0.031857326668530805
duration_ms 4246.843382671825


I tested with several columns and couldn't find a single column that doesn't show dependency. Therefore, I conducted a screening analysis using the code above. The result is that across all tested variables, including track_genre, explicit, release_year, danceability, key, mode, and time_signature, the permutation tests produced very small p-values. In each case, the observed test statistic was substantially larger than the values generated under the null distribution.

These results suggest that the missingness of tempo is associated with multiple observed variables in the dataset. Therefore, the missingness mechanism is unlikely to be MCAR. Instead, the evidence is more consistent with a MAR mechanism, where the probability that a tempo value is missing depends on other observed characteristics of the songs.

## Hypothesis Testing

**Null Hypothesis**: The distribution of `track_genre` is the same for popular and non-popular songs.
<br><br>
**Alternative Hypothesis**: The distribution of `track_genre` differs between popular and non-popular songs.
<br><br>
**Test Statistic**: TVD between the genre distributions of popular and non-popular songs.

In [204]:
# Observed TVD
observed_tvd = tvd(
    cleaned_tracks,
    'is_popular',
    'track_genre'
)

observed_tvd

np.float64(0.4786226789231056)

In [205]:
# Permutation Test
tvds = []

for _ in range(1000):

    shuffled = cleaned_tracks.copy()

    shuffled['is_popular'] = np.random.permutation(
        shuffled['is_popular']
    )

    tvds.append(
        tvd(
            shuffled,
            'is_popular',
            'track_genre'
        )
    )

tvds = np.array(tvds)

In [206]:
# P-value
p_value = np.mean(
    tvds >= observed_tvd
)

p_value

np.float64(0.0)

In [207]:
# Visualization
fig = px.histogram(
    x=tvds,
    nbins=30,
    title=f'Permutation Distribution (p = {p_value})'
)

fig.add_vline(
    x=observed_tvd,
    line_color='red',
    line_width=3,
    annotation_text='Observed TVD'
)

fig.show()

From this plot, we can see the observed TVD lies far outside the permutation distribution and p-value is approximately 0 as well. Therefore, I reject the null hypothesis and conclude that the genre distribution differs between popular and non-popular songs. This suggests that genre is associated with popularity and hence is an important indicator in the popularity classification task.

## Prediction Problem

Can we predict whether a Spotify track will be popular (popularity ≥ 55) using its audio features, genre, and metadata?

In [208]:
# Response variable
y = cleaned_tracks['is_popular']

Type: Binary Classification

True  = Popular song (popularity ≥ 55) <br>
False = Not popular song (popularity < 55)

## Baseline Model

In [209]:
from sklearn.model_selection import train_test_split

X = cleaned_tracks.drop(columns=['popularity', 'is_popular'])
y = cleaned_tracks['is_popular']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [210]:
baseline_features = [
    'danceability',
    'track_genre'
]

X_train_base = X_train[baseline_features]
X_test_base = X_test[baseline_features]

In [211]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression

preprocessor = ColumnTransformer(
    [
        ('genre', 
         OneHotEncoder(handle_unknown='ignore'), 
         ['track_genre'])
    ],
    remainder='passthrough'
)

baseline_model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000))
])

In [212]:
baseline_model.fit(X_train_base, y_train)

/Users/amelia/ENTER/envs/dsc80/lib/python3.10/site-packages/sklearn/compose/_column_transformer.py:1623: FutureWarning:


The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).




Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('genre',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['track_genre'])])),
                ('classifier', LogisticRegression(max_iter=1000))])

In [213]:
baseline_pred = baseline_model.predict(X_test_base)
baseline_pred

array([False, False, False, ..., False, False, False])

## Final Model

In [214]:
# final Features 
final_features = [
    'danceability',
    'energy',
    'loudness',
    'speechiness',
    'acousticness',
    'instrumentalness',
    'liveness',
    'valence',
    'tempo',
    'duration_min',
    'explicit',
    'release_year',
    'num_artists',
    'track_genre'
]

X_train_final = X_train[final_features]
X_test_final = X_test[final_features]

In [215]:
numeric_features = [
    'danceability',
    'energy',
    'loudness',
    'speechiness',
    'acousticness',
    'instrumentalness',
    'liveness',
    'valence',
    'tempo'
]

categorical_features = [
    'track_genre',
    'explicit'
]

I selected Random Forest as my final model because it can capture nonlinear relationships between audio features and popularity.

In [216]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

In [217]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(
        random_state=42
    ))
])

In [218]:
# Choose the best Hyperparameter

from sklearn.model_selection import GridSearchCV

param_grid = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [10, 20, None]
}

grid = GridSearchCV(
    pipe, 
    param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1
)

grid.fit(X_train_final, y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(transformers=[('num',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer(strategy='median'))]),
                                                                         ['danceability',
                                                                          'energy',
                                                                          'loudness',
                                                                          'speechiness',
                                                                          'acousticness',
                                                                          'instrumentalness',
                                                                          'liveness',
                                                                          'valence',
                                                                          'tempo']),
                                                                        ('cat',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer(strategy='most_frequent')),
                                                                                         ('encoder',
                                                                                          OneHotEncoder(handle_unknown='ignore'))]),
                                                                         ['track_genre',
                                                                          'explicit'])])),
                                       ('model',
                                        RandomForestClassifier(random_state=42))]),
             n_jobs=-1,
             param_grid={'model__max_depth': [10, 20, None],
                         'model__n_estimators': [100, 200]},
             scoring='f1')

In [219]:
# Best parameters
grid.best_params_

{'model__max_depth': None, 'model__n_estimators': 200}

In [220]:
final_preds = grid.predict(X_test_final)